In [1]:

import pandas as pd
import numpy as np

df = pd.read_csv("messy_customer_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset shape: (12, 7)


,Customer_ID,Name,Gender,Age,Join_Date,Spend,City
0,C001,Asha,Female,25.0,2024-01-05,120.50,Pune
1,C002,Rahul,M,34.0,05/02/2024,200.00,Mumbai
2,C003,Meera,female,NaN,2024/03/12,175.25,pune
3,C004,Imran,Male,29.0,14-04-2024,NaN,Delhi
4,C005,Priya,F,42.0,2024-05-20,310.00,Mumbai


In [2]:

# 1. Dataset overview
print("Dataset Shape:", df.shape)

# 2. Missing values per column
print("\nMissing Values Per Column:")
display(df.isnull().sum().to_frame("Missing Values"))

# 3. Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# 4. Data types
print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

# 5. Numeric columns summary
print("\nNumeric Summary:")
display(df.describe(include="all").T)


Dataset Shape: (12, 7)

Missing Values Per Column:


,Missing Values
Customer_ID,0
Name,0
Gender,0
Age,3
Join_Date,0
Spend,1
City,1



Duplicate Rows: 1

Data Types:


,Data Type
Customer_ID,str
Name,str
Gender,str
Age,float64
Join_Date,str
Spend,float64
City,str



Numeric Summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Customer_ID,12,11,C003,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Name,12,11,Meera,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,12,7,Female,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,9.0,NaN,NaN,NaN,44.222222,40.074237,24.0,27.0,31.0,36.0,150.0
Join_Date,12,11,2024/03/12,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Spend,11.0,NaN,NaN,NaN,9261.363636,30094.325463,95.0,167.625,180.0,230.0,99999.0
City,11,6,Pune,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:

print("Column Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())


Column Names:
['Customer_ID', 'Name', 'Gender', 'Age', 'Join_Date', 'Spend', 'City']

Missing Values:
Customer_ID    0
Name           0
Gender         0
Age            3
Join_Date      0
Spend          1
City           1
dtype: int64



## Missing Data Handling

Missing values will be handled according to the data type and meaning of each column. Numerical columns may use median imputation, categorical columns may use mode imputation, and rows will only be deleted when justified. Each decision will be documented to preserve data quality.


In [4]:
print(df.isnull().sum())
print("\nColumn names:")
print(df.columns.tolist())

Customer_ID    0
Name           0
Gender         0
Age            3
Join_Date      0
Spend          1
City           1
dtype: int64

Column names:
['Customer_ID', 'Name', 'Gender', 'Age', 'Join_Date', 'Spend', 'City']



## Missing Data Handling — Justification

- Age: Missing values are filled with the median because it is less affected by extreme values.
- Spend: Missing values are filled with the median to reduce the influence of unusually high spending values.
- City: Missing values are replaced with "Unknown" to preserve customer records without guessing the city.
- Other columns: No imputation is required because they contain no missing values.


In [5]:

# Handle missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Spend"] = df["Spend"].fillna(df["Spend"].median())
df["City"] = df["City"].fillna("Unknown")

# Verify the result
print("Missing values after handling:")
print(df.isnull().sum())

print("\nTotal missing values remaining:", df.isnull().sum().sum())


Missing values after handling:
Customer_ID    0
Name           0
Gender         0
Age            0
Join_Date      0
Spend          0
City           0
dtype: int64

Total missing values remaining: 0


In [6]:

print("Duplicate rows before removal:", df.duplicated().sum())


Duplicate rows before removal: 1


In [7]:

# Count duplicates before removal
duplicates_before = df.duplicated().sum()

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Count duplicates after removal
duplicates_after = df.duplicated().sum()

print("Duplicates before cleaning:", duplicates_before)
print("Duplicates removed:", duplicates_before - duplicates_after)
print("Duplicates remaining:", duplicates_after)
print("Rows remaining:", len(df))


Duplicates before cleaning: 1
Duplicates removed: 1
Duplicates remaining: 0
Rows remaining: 11


In [8]:

# Check existing values before standardisation
print("Gender values:")
print(df["Gender"].value_counts(dropna=False))

print("\nCity values:")
print(df["City"].value_counts(dropna=False))


Gender values:
Gender
Female    2
M         2
Male      2
F         2
female    1
male      1
FEMALE    1
Name: count, dtype: int64

City values:
City
Pune       3
Mumbai     2
Delhi      2
pune       1
DELHI      1
Unknown    1
mumbai     1
Name: count, dtype: int64


In [9]:

# Standardise Gender
df["Gender"] = (
    df["Gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "f": "Female",
        "female": "Female",
        "m": "Male",
        "male": "Male"
    })
)

# Standardise City
df["City"] = (
    df["City"]
    .astype("string")
    .str.strip()
    .str.title()
)

# Verify the changes
print("Gender after standardisation:")
print(df["Gender"].value_counts(dropna=False))

print("\nCity after standardisation:")
print(df["City"].value_counts(dropna=False))


Gender after standardisation:
Gender
Female    6
Male      5
Name: count, dtype: int64[pyarrow]

City after standardisation:
City
Pune       4
Mumbai     3
Delhi      3
Unknown    1
Name: count, dtype: int64[pyarrow]


In [10]:

# Correct data types
df["Customer_ID"] = df["Customer_ID"].astype("string")
df["Name"] = df["Name"].astype("string")
df["Join_Date"] = pd.to_datetime(df["Join_Date"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce").astype("Int64")
df["Spend"] = pd.to_numeric(df["Spend"], errors="coerce")

print("Corrected data types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDate conversion check:")
print(df["Join_Date"].head())
print("Invalid or missing dates:", df["Join_Date"].isna().sum())


Corrected data types:


,Data Type
Customer_ID,string
Name,string
Gender,string
Age,Int64
Join_Date,datetime64[us]
Spend,float64
City,string



Date conversion check:
0   2024-01-05
1          NaT
2          NaT
3          NaT
4   2024-05-20
Name: Join_Date, dtype: datetime64[us]
Invalid or missing dates: 6


In [11]:

# Reload original dates from the CSV
original_dates = pd.read_csv("messy_customer_dataset.csv")

print("Original Join_Date values:")
display(original_dates[["Join_Date"]])

print("\nDates that could not be parsed:")
display(
    original_dates.loc[
        df["Join_Date"].isna().reindex(original_dates.index, fill_value=False),
        ["Join_Date"]
    ]
)


Original Join_Date values:


,Join_Date
0,2024-01-05
1,05/02/2024
2,2024/03/12
3,14-04-2024
4,2024-05-20
5,06/15/2024
6,2024-07-02
7,2024/08/10
8,09-12-2024
9,2024-10-01



Dates that could not be parsed:


,Join_Date
1,05/02/2024
2,2024/03/12
3,14-04-2024
5,06/15/2024
7,2024/08/10
8,09-12-2024


In [12]:
print(original_dates["Join_Date"].to_list())

['2024-01-05', '05/02/2024', '2024/03/12', '14-04-2024', '2024-05-20', '06/15/2024', '2024-07-02', '2024/08/10', '09-12-2024', '2024-10-01', '2024/03/12', '2024-12-18']


In [13]:
print(original_dates["Join_Date"].to_list())

['2024-01-05', '05/02/2024', '2024/03/12', '14-04-2024', '2024-05-20', '06/15/2024', '2024-07-02', '2024/08/10', '09-12-2024', '2024-10-01', '2024/03/12', '2024-12-18']


In [14]:

# Reload the original dates as strings
original_dates = pd.read_csv(
    "messy_customer_dataset.csv",
    dtype={"Join_Date": "string"}
)

# Parse the supported date formats
date_formats = [
    "%Y-%m-%d",
    "%d/%m/%Y",
    "%Y/%m/%d",
    "%d-%m-%Y",
    "%m/%d/%Y",
    "%m-%d-%Y"
]

def parse_mixed_date(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    for fmt in date_formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue

    return pd.NaT

df["Join_Date"] = original_dates["Join_Date"].apply(parse_mixed_date)

print("Converted dates:")
display(df[["Join_Date"]])

print("Invalid dates remaining:", df["Join_Date"].isna().sum())


Converted dates:


,Join_Date
0,2024-01-05
1,2024-02-05
2,2024-03-12
3,2024-04-14
4,2024-05-20
5,2024-06-15
6,2024-07-02
7,2024-08-10
8,2024-12-09
9,2024-10-01


Invalid dates remaining: 0



## Date Standardisation — Decision

The Join_Date column contained multiple date formats. Dates were converted into a consistent datetime format using explicit format parsing. Ambiguous slash-separated dates were interpreted as day/month/year unless that interpretation was invalid, in which case month/day/year was used. Invalid dates would be investigated rather than silently replaced.


In [15]:

# Detect outliers using the IQR method
for col in ["Age", "Spend"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"\nColumn: {col}")
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Outlier count:", len(outliers))
    display(outliers[[col]])



Column: Age
Lower bound: 17.5
Upper bound: 45.5
Outlier count: 1


,Age
6,150



Column: Spend
Lower bound: 74.0625
Upper bound: 323.5625
Outlier count: 1


,Spend
6,99999.0



## Outlier Detection and Treatment

The IQR method was used to identify potential outliers in the Age and Spend columns.

- Age: Values will be checked for unrealistic ages. Valid extreme values will be retained, while invalid ages will be investigated.
- Spend: Unusually high spending values will be reviewed before deciding whether capping is appropriate.
- Outlier treatment decisions will be based on the data and business context rather than automatically deleting records.


In [16]:

# Reload original dataset for a fair before-and-after comparison
original = pd.read_csv("messy_customer_dataset.csv")

# Summary before cleaning
before_summary = {
    "Row Count": len(original),
    "Total Nulls": int(original.isnull().sum().sum()),
    "Duplicate Rows": int(original.duplicated().sum())
}

# Summary after cleaning
after_summary = {
    "Row Count": len(df),
    "Total Nulls": int(df.isnull().sum().sum()),
    "Duplicate Rows": int(df.duplicated().sum())
}

comparison = pd.DataFrame({
    "Before Cleaning": before_summary,
    "After Cleaning": after_summary
})

display(comparison)

print("\nFinal data types:")
display(df.dtypes.to_frame("Data Type"))

# Save cleaned dataset
df.to_csv("cleaned_customer_dataset.csv", index=False)

print("\nCleaned dataset saved successfully!")


,Before Cleaning,After Cleaning
Row Count,12,11
Total Nulls,5,0
Duplicate Rows,1,0



Final data types:


,Data Type
Customer_ID,string
Name,string
Gender,string
Age,Int64
Join_Date,datetime64[us]
Spend,float64
City,string



Cleaned dataset saved successfully!


In [17]:

print("Final dataset shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
print("Remaining duplicate rows:", df.duplicated().sum())
print("Invalid dates:", df["Join_Date"].isna().sum())

display(df.head())


Final dataset shape: (11, 7)
Remaining missing values: 0
Remaining duplicate rows: 0
Invalid dates: 0


,Customer_ID,Name,Gender,Age,Join_Date,Spend,City
0,C001,Asha,Female,25,2024-01-05,120.50,Pune
1,C002,Rahul,Male,34,2024-02-05,200.00,Mumbai
2,C003,Meera,Female,31,2024-03-12,175.25,Pune
3,C004,Imran,Male,29,2024-04-14,180.00,Delhi
4,C005,Priya,Female,42,2024-05-20,310.00,Mumbai


In [18]:

# Detailed before-and-after comparison

before = pd.read_csv("messy_customer_dataset.csv")
after = df.copy()

summary = pd.DataFrame({
    "Before Cleaning": [
        len(before),
        int(before.isnull().sum().sum()),
        int(before.duplicated().sum()),
        int(before.dtypes.notna().sum())
    ],
    "After Cleaning": [
        len(after),
        int(after.isnull().sum().sum()),
        int(after.duplicated().sum()),
        int(after.dtypes.notna().sum())
    ]
}, index=[
    "Row Count",
    "Total Missing Values",
    "Duplicate Rows",
    "Columns with a Data Type"
])

display(summary)


,Before Cleaning,After Cleaning
Row Count,12,11
Total Missing Values,5,0
Duplicate Rows,1,0
Columns with a Data Type,7,7


In [19]:

df.to_csv("cleaned_customer_dataset.csv", index=False)

# Verify the saved file
check_df = pd.read_csv("cleaned_customer_dataset.csv")

print("Saved file verified!")
print("Shape:", check_df.shape)
print("Missing values:", check_df.isnull().sum().sum())
print("Duplicate rows:", check_df.duplicated().sum())


Saved file verified!
Shape: (11, 7)
Missing values: 0
Duplicate rows: 0


In [20]:
for col in ["Age", "Spend"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    print(f"{col} outliers: {count}")

Age outliers: 1
Spend outliers: 1


In [21]:

for col in ["Age", "Spend"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df.loc[
        (df[col] < lower) | (df[col] > upper),
        ["Customer_ID", col]
    ]

    print(f"\n{col} outliers:")
    display(outliers)



Age outliers:


,Customer_ID,Age
6,C007,150



Spend outliers:


,Customer_ID,Spend
6,C007,99999.0



## Outlier Treatment — Decision

The IQR method identified one potential outlier in Age and one in Spend.

Outliers are not automatically errors. Their values and business context must be reviewed before treatment. Valid customer values will be retained, while impossible or clearly invalid values will be corrected using a justified method. Any capping or removal will be documented.



## Outlier Treatment and Justification

- Age: Customer C007 has an age of 150, which is considered unrealistic for this dataset. It is treated as invalid and replaced with the median age.
- Spend: Customer C007 has a spending value of 99999, which is unusually high compared with other customers. Since the value may be a data-entry error, it is capped at the upper IQR bound to reduce its influence while preserving the customer record.
- The customer row is retained rather than deleted.


In [22]:

# Correct invalid age
age_median = df.loc[df["Age"].between(0, 100), "Age"].median()
df.loc[~df["Age"].between(0, 100), "Age"] = age_median

# Cap extreme spending using the IQR upper bound
q1 = df["Spend"].quantile(0.25)
q3 = df["Spend"].quantile(0.75)
iqr = q3 - q1

upper_bound = q3 + 1.5 * iqr

df["Spend"] = df["Spend"].clip(upper=upper_bound)

print("Corrected C007 record:")
display(df.loc[df["Customer_ID"] == "C007"])


Corrected C007 record:


,Customer_ID,Name,Gender,Age,Join_Date,Spend,City
6,C007,Neha,Female,31,2024-07-02,323.5625,Pune


In [23]:

# Save the final cleaned dataset
df.to_csv("cleaned_customer_dataset.csv", index=False)

# Final validation
print("Final dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Invalid dates:", df["Join_Date"].isna().sum())

display(df.head())


Final dataset shape: (11, 7)
Missing values: 0
Duplicate rows: 0
Invalid dates: 0


,Customer_ID,Name,Gender,Age,Join_Date,Spend,City
0,C001,Asha,Female,25,2024-01-05,120.50,Pune
1,C002,Rahul,Male,34,2024-02-05,200.00,Mumbai
2,C003,Meera,Female,31,2024-03-12,175.25,Pune
3,C004,Imran,Male,29,2024-04-14,180.00,Delhi
4,C005,Priya,Female,42,2024-05-20,310.00,Mumbai
